# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# Add a new column
df['revenue'] = df['qty'] * df['price']

# calculate sum
revenue_sum = df['revenue'].sum()

# print information
print("shape (rows, columns):",df.shape)
print(f"revenue total: ${revenue_sum}")

shape (rows, columns): (400, 5)
revenue total: $8520.0


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# categorize revenue by category
by_category = df.groupby('category', as_index=False)['revenue'].sum()

# add share of total to revenue_summary
by_category['share_of_total'] = ((by_category['revenue']/revenue_sum) * 100).round(2)
by_category

,category,revenue,share_of_total
0,Drink,1554.0,18.24
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
3,RainGear,901.5,10.58


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# group orders by vendors
# then add column for how many orders that vendor made
# then add column for the average revenue of that vendor
vendor_summary = df.groupby('vendor_id',as_index = False).agg(
    order_count = ('qty','count'),
    avg_revenue = ('revenue','mean'),
).round(2)
vendor_summary


,vendor_id,order_count,avg_revenue
0,V-01,94,22.60
1,V-05,93,20.58
2,V-10,105,20.31
3,V-18,108,21.75


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# filter categories to only be merch, and sum the revenue for that filter
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_percentage = ((merch_revenue/revenue_sum) * 100.0).round(1)
print(f"Percent of revenue that comes from merch: {merch_percentage}%")


Percent of revenue that comes from merch: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# merge vendor_names with the original dataframe
joined = df.merge(vendor_names, on = 'vendor_id', how = 'left',validate = 'many_to_one', indicator = True)

# Compare old and new row count
print("original shape (rows, columns) = ",df.shape)
print("new shape (rows, columns) = ", joined.shape)
print("rows before and after are equal: ", df.shape[0] == joined.shape[0])
print("\n")

# Find/report unknown vendor
missing = joined[joined['_merge'] == 'left_only']
print("unmatched vendor(s):", missing['vendor_id'].unique())
print(f'unmatched orders: {len(missing)} | '
      f'revenue at stake: ${missing["revenue"].sum():.2f}')
print("\n")

# decide what to do with unknown vendor (I decided to label it as unknown and keep it)
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')

# Compare old and new revenue total
print(f"original revenue total: ${revenue_sum}")
print(f"new revenue total: ${joined['revenue'].sum()}")
print("new and old revenue totals are the same: ", revenue_sum == joined['revenue'].sum())

# TODO: merge, validate, and report the unmatched vendor

original shape (rows, columns) =  (400, 5)
new shape (rows, columns) =  (400, 7)
rows before and after are equal:  True


unmatched vendor(s): ['V-18']
unmatched orders: 108 | revenue at stake: $2349.00


original revenue total: $8520.0
new revenue total: $8520.0
new and old revenue totals are the same:  True


**The unmatched vendor, and what I did about it:** I identified the unknown vendor and reported it. I then decided to keep the unknown vendor and give it a readable name so that the revenue total would be preserved.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# use pivot_table() function to create it
report = joined.pivot_table(
    index = 'vendor_name',
    columns = 'category',
    values = 'revenue',
    aggfunc = 'sum',
    fill_value = 0,
    margins = True,
    margins_name = 'Total'
)
report

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

Since Food accounts for 50.39% of total revenue, I would make sure all food vendors are well stocked and ready for the high demand that seems to occur. Additionally, all four vendors have very equal order counds and revenue averages with order counts being (94, 93, 105, 108) which has a range of 15, and revenue averages being \$(22.60, 20.58, 20.31, 21.75)
 which has a range of \$2.29. Due to this I would make sure to keep all of the vendors since they seem to have similar order volumes and average order revenue. Of the seven answers given, the least trustworthy answer is likely the vendor analysis since one vendor ID was unknown and not included in the vendor lookup. I kept those orders and labeled them to preserve the revenu total, but cannot identify who the revenue belongs to. Due to that, it is harder to make specific conclusions comparing specific named vendors.